# PENPAL — Creativity, Novelty, and Reader Appreciation

A self-contained, tweakable notebook that reproduces every measure used in the
*"Productive Asymmetry in Collaborative Writing"* analysis, computed **from raw text**:

1. **Story-level human ratings** and composites (from `penpal_annotations_final.csv`).
2. **Stylometry** — 9+ transparent surface features.
3. **Topic-based** novelty / transience / resonance / entropy (Barron-style KL over **LDA topic** distributions).
4. **Word-level KL** novelty / transience / resonance (Barron-style KL over **smoothed unigram** distributions).
5. **Surprisal-based** novelty / transience / resonance over fixed-length **windows** (any HF causal LM, default `distilgpt2`).
6. **Analyses** — condition comparisons, reliability, correlations, and figures.

Everything is driven by the **Configuration** cell below: change a value, re-run, done.

**Requirements:** `numpy pandas scipy scikit-learn matplotlib`; for section 5 also `torch transformers`
(the LM downloads on first use). If you don't have them, set `RUN_SURPRISAL = False`.


## 0. Imports

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib torch transformers tqdm accelerate

import re, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 1. Configuration — *tweak these and re-run*

In [ ]:
ROOT        = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR    = ROOT / "data" / "annotations"
DATA_PATH   = DATA_DIR / "penpal_annotations_final.csv"   # annotations with a `condition` column
RANDOM_SEED = 0

# --- Topic-based (Barron on LDA topics) ---
N_TOPICS     = 25     # number of LDA topics
TOPIC_WINDOW = 3      # neighbour window (in sentences) for KL
LDA_MAX_ITER = 25

# --- Word-level KL (Barron on words) ---
WINDOW_WORDS  = 28    # window length in words
KL_ALPHA      = 0.01  # Laplace smoothing for unigram distributions
KL_NEIGHBOURS = 5     # neighbour window (in windows) for KL

# --- Surprisal (LM-based, on windows) ---
RUN_SURPRISAL       = True
SURPRISAL_MODEL     = "google/gemma-4-31b"   # any HF causal LM; larger = slower but richer
SURPRISAL_WINDOW    = 28             # window length in WORDS (~ mean words/turn in the source corpus)
SURPRISAL_WINDOW_GRID = [14, 28, 56, 112]  # robustness sweep over window size (Section 5d)

CONDITION_MAP   = {"hh": "HH", "ha": "H-LLM", "aa": "LLM-LLM"}
CONDITION_ORDER = ["HH", "H-LLM", "LLM-LLM"]
LLMNESS         = {"HH": 0, "H-LLM": 1, "LLM-LLM": 2}
np.random.seed(RANDOM_SEED)

### Helper functions (KL divergence and Benjamini–Hochberg FDR)

In [3]:
def kl(p, q, eps=1e-12):
    "Kullback–Leibler divergence in bits, KL(p || q), with additive floor."
    p = np.asarray(p) + eps; q = np.asarray(q) + eps
    return float(np.sum(p * np.log2(p / q)))

def bh(pvals):
    "Benjamini–Hochberg FDR-adjusted p-values."
    p = np.asarray(pvals, float); order = p.argsort()
    ranks = np.empty_like(order); ranks[order] = np.arange(1, len(p) + 1)
    q = p * len(p) / ranks
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    out = np.empty_like(q); out[order] = q_sorted
    return np.clip(out, 0, 1)

## 2. Load data & build the story-level table
Two annotators rate each story on six 1–5 dimensions; we average to the story level and
add three composites (coherence, creativity, attractiveness) plus an overall mean.

In [4]:
RENAME = {"coherence_element_consistency": "consistency", "coherence_logical_progression": "coherence",
          "creativity_originality": "originality", "creativity_surprisingness": "surprisingness",
          "likeability_enjoyability": "enjoyment", "likeability_quality": "quality"}
DIMS = list(RENAME.values())

ann = pd.read_csv(DATA_PATH).rename(columns=RENAME)
for d in DIMS:
    ann[d] = pd.to_numeric(ann[d], errors="coerce")
ann["cond"] = ann["condition"].astype(str).str.lower().map(CONDITION_MAP)

story = ann.groupby("id").agg(cond=("cond", "first"), text=("text", "first"),
                              n_annot=("annotator", "nunique"),
                              **{d: (d, "mean") for d in DIMS})
story["coherence_comp"]  = story[["consistency", "coherence"]].mean(axis=1)
story["creativity_comp"] = story[["originality", "surprisingness"]].mean(axis=1)
story["attract_comp"]    = story[["enjoyment", "quality"]].mean(axis=1)
story["overall"]         = story[DIMS].mean(axis=1)
story["llm"]             = story["cond"].map(LLMNESS)

print("Stories per condition:")
print(story["cond"].value_counts().reindex(CONDITION_ORDER))
story[["cond"] + DIMS].head()

Stories per condition:
cond
HH         36
H-LLM      91
LLM-LLM    20
Name: count, dtype: int64


,cond,consistency,coherence,originality,surprisingness,enjoyment,quality
id,,,,,,,
3,LLM-LLM,5.0,5.0,5.0,5.0,4.0,5.0
6,LLM-LLM,5.0,5.0,1.5,4.0,1.0,4.0
8,LLM-LLM,5.0,5.0,5.0,5.0,5.0,5.0
9,LLM-LLM,4.0,4.5,2.5,4.0,4.5,4.0
10,LLM-LLM,5.0,5.0,2.0,5.0,4.5,5.0


## 3. Stylometric features
Transparent surface features computed directly from each completed story, plus a
corpus-relative *semantic distinctiveness* (1 − cosine to the nearest other story).

In [5]:
WORD = re.compile(r"[A-Za-z']+")
def _syll(w):
    w = w.lower(); n = len(re.findall(r"[aeiouy]+", w))
    return max(1, n - (1 if w.endswith("e") else 0))

def stylometry(text):
    ws = WORD.findall(str(text))
    sents = [s for s in re.split(r"(?<=[.!?])\s+", str(text).strip()) if s.split()]
    nw, ns = len(ws), max(1, len(sents)); low = [w.lower() for w in ws]; uniq = set(low)
    hap = sum(1 for w in uniq if low.count(w) == 1)
    out = {"n_words": nw, "n_sentences": len(sents),
           "mean_word_length": np.mean([len(w) for w in ws]) if ws else 0,
           "mean_sentence_length": nw / ns, "ttr": len(uniq) / nw if nw else 0,
           "hapax_ratio": hap / nw if nw else 0,
           "comma_density": str(text).count(",") / nw * 1000 if nw else 0,
           "flesch_approx": 206.835 - 1.015 * (nw / ns) - 84.6 * (sum(_syll(w) for w in ws) / nw if nw else 0)}
    try:
        if len(sents) >= 2:
            S = cosine_similarity(TfidfVectorizer().fit_transform(sents))
            out["adjacent_sentence_similarity"] = float(np.mean([S[i, i + 1] for i in range(len(sents) - 1)]))
        else:
            out["adjacent_sentence_similarity"] = np.nan
    except Exception:
        out["adjacent_sentence_similarity"] = np.nan
    return out

feat = pd.DataFrame({i: stylometry(t) for i, t in tqdm(story["text"].items(), desc="Stylometry")}).T
Vt = TfidfVectorizer(ngram_range=(1, 2), min_df=2).fit_transform(story["text"].tolist())
Sim = cosine_similarity(Vt); np.fill_diagonal(Sim, 0)
feat["semantic_distinctiveness"] = 1 - Sim.max(axis=1)
story = story.join(feat)
feat.describe().round(3)

,n_words,n_sentences,mean_word_length,mean_sentence_length,ttr,hapax_ratio,comma_density,flesch_approx,adjacent_sentence_similarity,semantic_distinctiveness
count,147.000,147.000,147.000,147.000,147.000,147.000,147.000,147.000,147.000,147.000
mean,504.293,29.599,4.458,20.282,0.509,0.373,66.398,65.391,0.087,0.611
std,74.158,10.016,0.262,14.183,0.047,0.052,20.857,17.029,0.040,0.073
min,393.000,5.000,3.880,9.204,0.395,0.231,9.324,-46.401,0.031,0.226
25%,459.000,23.000,4.250,14.181,0.475,0.342,50.407,59.894,0.063,0.576
50%,489.000,29.000,4.439,16.517,0.512,0.375,69.114,67.781,0.078,0.623
75%,538.500,35.000,4.646,20.060,0.543,0.411,80.838,75.520,0.097,0.660
max,880.000,59.000,5.180,121.800,0.607,0.486,114.754,87.946,0.275,0.744


## 4. Topic-based novelty / transience / resonance (Barron on LDA topics)
Fit **one** LDA over all sentences; represent each sentence by its topic mixture $\theta$.
Within a story, for a neighbour window $w$:
$$\text{Novelty}_j = \tfrac1w\!\sum_{i=1}^{w}\mathrm{KL}(\theta_j\,\|\,\theta_{j-i}),\quad
\text{Transience}_j = \tfrac1w\!\sum_{i=1}^{w}\mathrm{KL}(\theta_j\,\|\,\theta_{j+i}),\quad
\text{Resonance}_j = \text{Novelty}_j-\text{Transience}_j.$$
`topic_entropy` is the mean entropy of the per-sentence topic mixture.

In [6]:
SENT = re.compile(r"(?<=[.!?])\s+")
def split_sentences(t): return [s for s in SENT.split(str(t).strip()) if len(s.split()) >= 3]

story_sents = {i: split_sentences(t) for i, t in story["text"].items()}
all_sents = [s for ss in story_sents.values() for s in ss]

cv  = CountVectorizer(stop_words="english", min_df=3)
X   = cv.fit_transform(all_sents)
lda = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=LDA_MAX_ITER,
                                learning_method="batch", random_state=RANDOM_SEED)
theta = lda.fit_transform(X)
theta = theta / theta.sum(axis=1, keepdims=True)

def topic_ntr(th, w=TOPIC_WINDOW):
    n = len(th); nov = np.full(n, np.nan); tra = np.full(n, np.nan)
    ent = -np.sum((th + 1e-12) * np.log2(th + 1e-12), axis=1)
    for j in range(n):
        past = range(max(0, j - w), j); fut = range(j + 1, min(n, j + 1 + w))
        if past: nov[j] = np.mean([kl(th[j], th[d]) for d in past])
        if fut:  tra[j] = np.mean([kl(th[j], th[d]) for d in fut])
    res = nov - tra
    return dict(topic_novelty=np.nanmean(nov), topic_transience=np.nanmean(tra),
                topic_resonance=np.nanmean(res), topic_entropy=float(np.mean(ent)))

rows, k = {}, 0
for i, ss in tqdm(story_sents.items(), desc="Topic KL"):
    th = theta[k:k + len(ss)]; k += len(ss)
    rows[i] = topic_ntr(th) if len(ss) >= 2 else dict.fromkeys(
        ["topic_novelty", "topic_transience", "topic_resonance", "topic_entropy"], np.nan)
story = story.join(pd.DataFrame(rows).T)
story[["topic_novelty", "topic_transience", "topic_resonance", "topic_entropy"]].describe().round(3)

,topic_novelty,topic_transience,topic_resonance,topic_entropy
count,147.000,147.000,147.000,147.000
mean,4.903,4.943,-0.057,1.803
std,0.782,0.840,0.339,0.284
min,1.815,1.820,-2.956,0.806
25%,4.402,4.435,-0.105,1.599
50%,4.825,4.837,-0.014,1.806
75%,5.375,5.362,0.079,2.001
max,7.505,8.677,0.477,2.391


## 5b. Word-level KL novelty / transience / resonance (Barron on words)
Identical construction to Section 4, but each unit is a fixed-length **word window**
represented by its **smoothed unigram** distribution instead of an LDA topic mixture —
no topic model required.

In [7]:
def word_windows(text, n=WINDOW_WORDS):
    toks = str(text).split()
    return [" ".join(toks[i:i + n]) for i in range(0, len(toks), n) if len(toks[i:i + n]) >= 3]

def word_ntr(text, w=KL_NEIGHBOURS, alpha=KL_ALPHA):
    units = word_windows(text)
    vocab = sorted({t for u in units for t in WORD.findall(u.lower())})
    idx = {v: i for i, v in enumerate(vocab)}
    P = np.zeros((len(units), len(vocab)))
    for r, u in enumerate(units):
        for t in WORD.findall(u.lower()): P[r, idx[t]] += 1
    P += alpha; P /= P.sum(axis=1, keepdims=True)
    n = len(units); nov = np.full(n, np.nan); tra = np.full(n, np.nan)
    for j in range(n):
        past = range(max(0, j - w), j); fut = range(j + 1, min(n, j + 1 + w))
        if past: nov[j] = np.mean([kl(P[j], P[d]) for d in past])
        if fut:  tra[j] = np.mean([kl(P[j], P[d]) for d in fut])
    res = nov - tra
    return dict(word_novelty=np.nanmean(nov), word_transience=np.nanmean(tra), word_resonance=np.nanmean(res))

story = story.join(pd.DataFrame({i: word_ntr(t) for i, t in tqdm(story["text"].items(), desc="Word KL")}).T)
story[["word_novelty", "word_transience", "word_resonance"]].describe().round(3)

,word_novelty,word_transience,word_resonance
count,147.000,147.000,147.000
mean,4.837,4.784,0.046
std,0.188,0.189,0.089
min,4.350,4.352,-0.239
25%,4.721,4.647,-0.000
50%,4.851,4.776,0.047
75%,4.958,4.898,0.101
max,5.372,5.338,0.320


## 5c. Surprisal-based novelty / transience / resonance (windows, any HF causal LM)
For a window $T_t$ with preceding story context $c_{<t}$ and immediately following window $F_{t+1}$:
$$\text{Novelty}_t=\bar s(T_t\mid c_{<t})-\bar s(T_t\mid \text{BOS}),\quad
\text{Transience}_t=\bar s(F_{t+1}\mid c_{<t},T_t)-\bar s(F_{t+1}\mid c_{<t}),\quad
\text{Resonance}_t=\text{Novelty}_t-\text{Transience}_t,$$
where $\bar s$ is mean token surprisal (bits). **Transience follows the updated (conditional/marginal)
definition from the EMNLP analysis**: it measures the marginal effect of $T_t$ on the surprisal of the
future window *given the context already established* ($c_{<t}$), rather than the older
$\bar s(F_{t+1}\mid T_t)-\bar s(F_{t+1}\mid\text{BOS})$ form that conditioned only on the single
current window. This makes novelty and transience share the same running-context construction.
Set `SURPRISAL_MODEL` to any causal LM. **This cell downloads the model on first run and is the slowest step.**

In [ ]:
if RUN_SURPRISAL:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from tqdm.auto import tqdm

    device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    tok = AutoTokenizer.from_pretrained(SURPRISAL_MODEL)
    try:
        lm = AutoModelForCausalLM.from_pretrained(
            SURPRISAL_MODEL,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None
        )
    except (ValueError, ImportError):
        lm = AutoModelForCausalLM.from_pretrained(
            SURPRISAL_MODEL,
            torch_dtype=dtype
        ).to(device)

    if not hasattr(lm, "hf_device_map") and device != "cpu":
        lm = lm.to(device)
    lm.eval()

    BOS = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    MAXCTX = getattr(lm.config, "n_positions", getattr(lm.config, "max_position_embeddings", 1024))

    @torch.no_grad()
    def _tok_surprisal(ids):
        ids = ids.to(device)
        logits = lm(ids.unsqueeze(0)).logits[0]
        logp = torch.log_softmax(logits[:-1].float(), dim=-1)
        return -logp.gather(1, ids[1:].unsqueeze(1)).squeeze(1) / math.log(2)

    def _mean_s(target, context=None):
        pre = torch.tensor([BOS], device=device)
        if context is not None and len(context) > 0:
            budget = MAXCTX - len(target) - 1
            if budget > 0: pre = torch.cat([pre, context[-budget:].to(device)])
        full = torch.cat([pre, target.to(device)])
        return _tok_surprisal(full)[-len(target):].mean().item()

    def _enc(txt):
        return tok(" " + txt.strip(), add_special_tokens=False, return_tensors="pt").input_ids[0]

    def surprisal_ntr(text, window=None):
        w = window or SURPRISAL_WINDOW
        toks = str(text).split()
        units = [" ".join(toks[i:i + w]) for i in range(0, len(toks), w)
                 if len(toks[i:i + w]) >= 3]
        ids = [_enc(u) for u in units]; n = len(units)
        if n < 2:
            return dict(s_novelty=np.nan, s_transience=np.nan, s_resonance=np.nan)
        base  = [_mean_s(ids[i]) for i in range(n)]                              # s(T_i | BOS)
        s_ctx = [base[0]] + [_mean_s(ids[i], torch.cat(ids[:i])) for i in range(1, n)]
        nov = np.full(n, np.nan); tra = np.full(n, np.nan)
        for i in range(1, n):                                                    # Novelty
            nov[i] = s_ctx[i] - base[i]
        for i in range(n - 1):                                                   # Transience (conditional)
            ctx_before = torch.cat(ids[:i]) if i > 0 else None                   # C_i  (excludes T_i)
            tra[i] = s_ctx[i + 1] - _mean_s(ids[i + 1], ctx_before)
        res = nov - tra
        return dict(s_novelty=np.nanmean(nov), s_transience=np.nanmean(tra), s_resonance=np.nanmean(res))

    rows = {}
    for i, t in tqdm(story["text"].items(), desc=f"Surprisal ({SURPRISAL_MODEL})"):
        rows[i] = surprisal_ntr(t)
    story = story.join(pd.DataFrame(rows).T)
    print(story[["s_novelty", "s_transience", "s_resonance"]].describe().round(3))
else:
    print("RUN_SURPRISAL = False — skipping LM surprisal (set it to True to compute).")


## 5d. Surprisal-window robustness sweep
Recompute the window Novelty/Transience/Resonance at $w\in\{14,28,56,112\}$ words and check whether
(i) the RQ1 monotone trend with LLM involvement and (ii) the novelty$\leftrightarrow$rating couplings
are stable across window size. This addresses the concern that $w=28$ (≈ mean words/turn) sits near the
worst case for windows straddling author transitions. Per-story values are saved to
`surprisal_window_sweep.csv`; the final `CROSS-WINDOW SUMMARY` table is the compact result to report.

In [ ]:
# --- Surprisal-window robustness sweep -------------------------------------------------
# Re-computes window Novelty/Transience/Resonance at several window sizes to test whether
# the RQ1 condition gradient and the novelty<->rating correlations are stable, or artefacts
# of the w=28 choice. w=28 ~ the mean words/turn in the source corpus, but windows are cut
# at fixed offsets and do NOT respect real turn boundaries, so a window can straddle an
# author transition (most consequential in H-LLM). If the gradient/correlations hold across
# window sizes, that concern is answered; if they move, the effect is windowing-dependent.
if RUN_SURPRISAL:
    SW_MEAS = ["s_novelty", "s_transience", "s_resonance"]
    sweep_rows, grid_summ = [], {}
    for w in tqdm(SURPRISAL_WINDOW_GRID, desc="Window sweep"):
        vals = {}
        for i, t in tqdm(story["text"].items(), desc=f"Window w={w}", leave=False):
            vals[i] = surprisal_ntr(t, window=w)
        wdf = pd.DataFrame(vals).T[SW_MEAS].join(story[["cond", "llm"] + DIMS])
        for i, r in wdf.iterrows():
            sweep_rows.append(dict(id=i, window=w, cond=r["cond"],
                                   **{m: r[m] for m in SW_MEAS}))
        trend = {m: spearmanr(wdf["llm"], wdf[m], nan_policy="omit")[0] for m in SW_MEAS}
        ratecorr = pd.DataFrame(
            {m: [spearmanr(wdf[m], wdf[d], nan_policy="omit")[0] for d in DIMS]
             for m in SW_MEAS}, index=DIMS).T
        cond_means = wdf.groupby("cond")[SW_MEAS].mean().reindex(CONDITION_ORDER)
        grid_summ[w] = dict(trend=trend, ratecorr=ratecorr, cond_means=cond_means)
        n_units = [len([1 for k in range(0, len(str(t).split()), w)
                        if len(str(t).split()[k:k + w]) >= 3]) for t in story["text"]]
        print(f"\n===== window = {w} words   (mean {np.mean(n_units):.1f} windows/story) " + "=" * 30)
        print("RQ1 trend with LLM-ness (rho):  " +
              "   ".join(f"{m}={trend[m]:+.3f}" for m in SW_MEAS))
        print("Condition means:"); print(cond_means.round(3))
        print("Spearman (measure x rating):"); print(ratecorr.round(2))

    sweep_long = pd.DataFrame(sweep_rows)
    sweep_long.to_csv(DATA_DIR / "surprisal_window_sweep.csv", index=False)

    # Compact cross-window comparison to paste back: RQ1 gradient + key rating couplings
    comp = pd.DataFrame(
        {w: {**{f"trend_{m}": grid_summ[w]["trend"][m] for m in SW_MEAS},
             "snov_x_originality": grid_summ[w]["ratecorr"].loc["s_novelty", "originality"],
             "snov_x_quality":     grid_summ[w]["ratecorr"].loc["s_novelty", "quality"],
             "snov_x_enjoyment":   grid_summ[w]["ratecorr"].loc["s_novelty", "enjoyment"],
             "sres_x_originality": grid_summ[w]["ratecorr"].loc["s_resonance", "originality"]}
         for w in SURPRISAL_WINDOW_GRID}).T
    comp.index.name = "window"
    print("\n\n===== CROSS-WINDOW SUMMARY (paste this back) =====")
    print(comp.round(3).to_string())
    print(f"\nSaved per-story sweep -> {DATA_DIR / 'surprisal_window_sweep.csv'}")
else:
    print("RUN_SURPRISAL = False — skipping window sweep.")

## 6. Human evaluations by condition
Means and Mann–Whitney planned contrasts (Benjamini–Hochberg FDR within each comparison).

In [ ]:
OUT = DIMS + ["overall"]
desc = story.groupby("cond")[OUT].mean().reindex(CONDITION_ORDER).round(2)
print("Condition means:\n", desc.T, "\n")
for a, b in [("H-LLM", "HH"), ("H-LLM", "LLM-LLM"), ("HH", "LLM-LLM")]:
    ps = [mannwhitneyu(story.loc[story.cond == a, m].dropna(),
                       story.loc[story.cond == b, m].dropna(),
                       alternative="two-sided").pvalue for m in OUT]
    print(f"{a} vs {b}  (p_FDR):")
    print("  " + "  ".join(f"{m[:5]}={q:.3f}" for m, q in zip(OUT, bh(ps))))

## 7. Inter-annotator reliability

In [ ]:
rel = {}
for d in DIMS:
    a, b = [], []
    for i, g in ann.groupby("id"):
        v = g.sort_values("annotation_id")[d].dropna().values
        if len(v) >= 2: a.append(v[0]); b.append(v[1])
    rel[d] = {"rho": spearmanr(a, b)[0], "n": len(a)}
pd.DataFrame(rel).T.round(3)

## 8. Which measures predict which judgments?
Pooled Spearman correlations of every novelty measure with the six ratings, plus the
monotonic trend with LLM involvement. This is the central dissociation: **topic novelty**
tracks *quality*, while **surprisal / topic entropy** track *originality*.

In [ ]:
MEAS = ["topic_novelty", "topic_entropy", "word_novelty"]
if RUN_SURPRISAL: MEAS += ["s_novelty", "s_transience", "s_resonance"]
MEAS = [m for m in MEAS if m in story.columns]

corr = pd.DataFrame({m: [spearmanr(story[m], story[d], nan_policy="omit")[0] for d in DIMS]
                     for m in MEAS}, index=DIMS).T
print("Spearman  (measure x rating):"); print(corr.round(2), "\n")
print("Trend with LLM-ness (HH < H-LLM < LLM-LLM):")
for m in MEAS:
    print(f"  {m:16s} rho = {spearmanr(story['llm'], story[m], nan_policy='omit')[0]:+.3f}")

## 9. Figures

In [ ]:
COL = {"HH": "#4C72B0", "H-LLM": "#DD8452", "LLM-LLM": "#55A868"}
pretty = ["Consistency", "Coherence", "Originality", "Surprisingness", "Enjoyment", "Quality"]

# (a) evaluation profile
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in CONDITION_ORDER:
    ax.plot(pretty, [story.loc[story.cond == c, d].mean() for d in DIMS],
            marker="o", label=c, color=COL[c], lw=2)
ax.set_ylabel("Mean rating (1–5)"); ax.legend(title="Condition", frameon=False)
ax.set_xticklabels(pretty, rotation=25, ha="right"); ax.set_title("Human evaluations by condition")
plt.tight_layout(); plt.show()

In [ ]:
# (b) quality–enjoyment map + measure x rating heatmap
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.2]})
for c in CONDITION_ORDER:
    s = story[story.cond == c]
    axL.scatter(s.enjoyment, s.quality, s=18, alpha=.5, color=COL[c], label=c)
    axL.scatter(s.enjoyment.mean(), s.quality.mean(), s=200, marker="X", color=COL[c], edgecolor="k", zorder=5)
axL.set_xlabel("Enjoyment"); axL.set_ylabel("Quality"); axL.legend(title="Condition", frameon=False)
axL.set_title("Quality–enjoyment plane")
H = np.array([[spearmanr(story[m], story[d], nan_policy="omit")[0] for d in DIMS] for m in MEAS])
im = axR.imshow(H, cmap="RdBu_r", vmin=-.5, vmax=.5, aspect="auto")
axR.set_xticks(range(6)); axR.set_xticklabels([d[:5] for d in DIMS], rotation=40, ha="right")
axR.set_yticks(range(len(MEAS))); axR.set_yticklabels(MEAS)
for i in range(len(MEAS)):
    for j in range(6): axR.text(j, i, f"{H[i, j]:+.2f}", ha="center", va="center", fontsize=7)
axR.set_title("Measure × rating (Spearman ρ)"); fig.colorbar(im, ax=axR, fraction=.046, pad=.04)
plt.tight_layout(); plt.show()

In [ ]:
# (c) window-surprisal dynamics by condition (only if computed)
if RUN_SURPRISAL:
    fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
    for ax, m, t in zip(axes, ["s_novelty", "s_transience", "s_resonance"], ["Novelty", "Transience", "Resonance"]):
        for j, c in enumerate(CONDITION_ORDER):
            v = story.loc[story.cond == c, m].dropna()
            ax.errorbar(j, v.mean(), yerr=1.96 * v.std() / np.sqrt(len(v)), fmt="o", color=COL[c], capsize=3)
        ax.set_title(t); ax.set_xticks(range(3)); ax.set_xticklabels(CONDITION_ORDER, rotation=25, ha="right")
    axes[0].set_ylabel("Window surprisal (bits)"); plt.tight_layout(); plt.show()

## 10. Save the full story-level table

In [ ]:
story.drop(columns=["text"]).to_csv(DATA_DIR / "penpal_measures_all.csv")
print("Saved -> {}  ({} stories, {} columns)".format(DATA_DIR / "penpal_measures_all.csv", *story.drop(columns=['text']).shape))

---
### How to extend
- **Different LM for surprisal:** set `SURPRISAL_MODEL` to any Hugging Face causal model (e.g. `"gpt2"`, `"gpt2-medium"`, or a local Llama path).
- **Topic granularity:** change `N_TOPICS` / `TOPIC_WINDOW`; re-run Section 4 onward.
- **Unit size:** `WINDOW_WORDS` / `SURPRISAL_WINDOW` control the word/token window length.
- **New corpus:** point `DATA_PATH` at any CSV with the same columns (`id`, `text`, `condition`, and the six rating columns).
- All measures are stored on the `story` DataFrame and exported to `penpal_measures_all.csv`.
